# 03. 실험 1 — 파라미터 복원 실험

앰프 계열 5 × 정답 θ 4 = 가짜 곡 20개 (EGDB test DI + 믹스 처리 증강).
비교: 시작점(prior_center 사전 범위 중심 / prior_multi 중심+샘플 / encoder 인코더 θ̂) × 통계(장기 평균 스펙트럼 / +레벨별 / 4종) × 세션(없음 none / 정확 correct / 오지정 wrong), 기준 (b) 세션만 경로.
대응표 v2 결과는 `runs/exp1_v2/exp1_results.jsonl`에(v1 결과 `runs/exp1/`은 그대로 둔다) 한 줄씩 쌓여서 중단 후 다시 실행하면 이어 간다.

In [ ]:
# 공통 준비: Drive 연결, 저장소 받기, 패키지 설치
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess
DRIVE = '/content/drive/MyDrive/song2nam'   # 데이터·실행 기록·체크포인트를 둘 Drive 폴더
REPO = '/content/song2nam'
BRANCH = 'main'                               # 작업 브랜치로 바꿔서 쓸 수 있음
os.makedirs(f'{DRIVE}/data', exist_ok=True)
os.makedirs(f'{DRIVE}/runs', exist_ok=True)
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-b', BRANCH, 'https://github.com/gmeag01/song2nam.git', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)
os.chdir(REPO)
# data/, runs/를 Drive에 연결해서 Colab 런타임이 끊겨도 남게 한다
for d in ['data', 'runs']:
    if os.path.islink(d):
        continue
    subprocess.run(['rm', '-rf', d])
    os.symlink(f'{DRIVE}/{d}', d)

In [ ]:
# Colab에는 torch/torchaudio가 이미 있으므로 나머지만 설치
!grep -vE '^(torch|torchaudio)$' requirements.txt > /tmp/req.txt && pip install -q -r /tmp/req.txt
import torch; print(torch.__version__, torch.cuda.is_available())

In [ ]:
import os
from ampcap.synth_data import di_reader_for_split
from ampcap.evaluate import Exp1Config, run_experiment1, summarize_exp1
EGDB, SPLIT = 'data/EGDB/audio_DI', 'data/egdb_split.json'
test_r = di_reader_for_split(EGDB, 'test', SPLIT)
pool_r = di_reader_for_split(EGDB, 'pool', SPLIT)
from ampcap.encoder import BEST_PATH, try_load
enc = try_load(BEST_PATH, 'cuda')   # 대응표 v2 인코더. 없거나 θ 버전이 다르면 None (시작점 prior_center·prior_sample_i만)
cfg = Exp1Config(steps=600)
# 대응표 v2는 끝까지 지키는 부분집합(all4)만 다시 돌린다(설계 문서 4.10). 통계 묶음 비교는 v1 결과로 보고한다
cfg.stat_sets = {'all4': ('ltas', 'level', 'flat', 'dyn')}
# ⑥ 드라이브 앵커 세기. 0 / 0.1 / 0.3을 결과 폴더를 나눠 돌리고 ESR과 게인 합 오차로 고른다(설계 4.10)
cfg.lambda_anchor = 0.1
OUT = 'runs/exp1_v2' if cfg.lambda_anchor == 0.1 else f'runs/exp1_v2_anchor{cfg.lambda_anchor}'


## 게인 식별 진단 (선택, 몇 분)


In [ ]:
# 게인 식별 진단(설계 4.7): 정답 θ에서 통계 민감도 JᵀJ의 가장 약한 방향 3개. 게인 파라미터가 몰리면 앵커·고정이 맞는 대응이다
from ampcap.evaluate import make_fake_songs, stat_sensitivity
from ampcap.graybox.amp import GrayboxAmp
from ampcap.synth_data import normalize_di
G = GrayboxAmp().to('cuda')
songs = make_fake_songs(test_r, cfg, G, 'cuda')
di = torch.from_numpy(normalize_di(pool_r.fixed_clips(4, 0))).to('cuda')
for s in songs[::4]:   # 계열마다 하나
    out = stat_sensitivity(s['theta'], di, G=G)
    print(s['id'], [(round(w['eigval'], 4), w['loadings']) for w in out['weak']])


In [ ]:
res = run_experiment1(test_r, pool_r, OUT, cfg, encoder_model=enc, device='cuda')

In [ ]:
df = summarize_exp1(f'{OUT}/exp1_results.jsonl')
df.to_csv(f'{OUT}/summary.csv'); df

## 끝까지 지키는 비교: 시작점 비교, 세션 정보 유무 비교

In [ ]:
import pandas as pd
from ampcap.evaluate import read_results
rows = read_results(f'{OUT}/exp1_results.jsonl')
d = pd.DataFrame([{k: v for k, v in r.items() if k != 'theta_err'} for r in rows])
a = d[d.stats.isin(['all4', '-'])].pivot_table(index='start', columns='session', values='esr', aggfunc='median')
a.to_csv(f'{OUT}/start_vs_session_esr.csv')
g = d[d.stats.isin(['all4', '-'])].pivot_table(index='start', columns='session', values='gain_total_err_db', aggfunc='median')
display(a); g   # 게인 6개 dB 합 오차(식별 가능 게인 조합)